# MediaPipe landmarks + SVM baseline (`mp-svm-001`)

Tối đa hai tay, landmarks chuẩn hóa theo từng tay, slot trái/phải, feature quan hệ hai cổ tay và RBF SVM. Dùng exact canonical dedup + participant split của `cnn-001`.


In [ ]:
%pip -q install 'mediapipe>=0.10.14' 'huggingface-hub>=0.25' pandas pyarrow scikit-learn matplotlib seaborn joblib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, random, re, urllib.request, zipfile
import cv2, joblib, mediapipe as mp, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from huggingface_hub import snapshot_download
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

EXPERIMENT_ID = 'mp-svm-001-participant-disjoint-rbf'
DATASET_REPO, DATASET_REVISION = 'hnam25/asl-hand-gesture-images', '8f36ac00ece6dfce94410a980a839d93a912d366'
RAW_ARCHIVE = 'ASL_HG_36000/ASL_Raw_Images.zip'
CNN_ARTIFACT_REPO, CNN_ARTIFACT_REVISION = 'hnam25/asl-hg-cnn-baseline', '064551d5634ef34d3e6a9132ab3d4a374b2f5633'
MEDIAPIPE_TASK_URL = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'
SEED, NUM_HANDS = 42, 2
CLASSES = [str(i) for i in range(10)] + [chr(i) for i in range(ord('A'), ord('Z') + 1)]
ROOT = Path('/content/asl-mp-svm'); HF_ROOT, CNN_ROOT, RAW, OUTPUTS = ROOT/'hf', ROOT/'cnn-artifact', ROOT/'raw', ROOT/'outputs'
for directory in (HF_ROOT, CNN_ROOT, RAW, OUTPUTS/'models', OUTPUTS/'metrics', OUTPUTS/'figures', OUTPUTS/'logs', OUTPUTS/'metadata'): directory.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)


In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()

snapshot_download(repo_id=DATASET_REPO, repo_type='dataset', revision=DATASET_REVISION, local_dir=HF_ROOT, allow_patterns=[RAW_ARCHIVE])
snapshot_download(repo_id=CNN_ARTIFACT_REPO, revision=CNN_ARTIFACT_REVISION, local_dir=CNN_ROOT, allow_patterns=['metadata/deduplication_manifest.csv', 'metadata/split_manifest.json', 'metadata/experiment_config.json'])
archive = HF_ROOT/RAW_ARCHIVE; cnn_config = json.loads((CNN_ROOT/'metadata'/'experiment_config.json').read_text())
if sha256_file(archive) != cnn_config['audit_manifest']['raw_archive_sha256']: raise RuntimeError('Raw archive SHA does not match cnn-001 audit provenance.')
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (RAW/member.filename).resolve()
        if RAW.resolve() not in target.parents and target != RAW.resolve(): raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
    z.extractall(RAW)
class_roots = [p.parent for p in RAW.rglob('0') if p.is_dir() and all((p.parent/label).is_dir() for label in CLASSES)]
if not class_roots: raise RuntimeError('Could not find raw class folders.')
CLASS_ROOT = sorted(class_roots, key=lambda p: len(p.parts))[0]
task_path = ROOT/'hand_landmarker.task'; urllib.request.urlretrieve(MEDIAPIPE_TASK_URL, task_path) if not task_path.exists() else None
task_sha256 = sha256_file(task_path)
print({'raw_archive_sha256': sha256_file(archive), 'task_sha256': task_sha256, 'class_root': str(CLASS_ROOT)})


In [ ]:
# Exact canonical sample set from cnn-001, now mapped back to raw images.
dedup = pd.read_csv(CNN_ROOT/'metadata'/'deduplication_manifest.csv'); split_manifest = json.loads((CNN_ROOT/'metadata'/'split_manifest.json').read_text())
canonical = dedup[dedup.is_canonical.astype(str).str.lower().eq('true')].copy()
canonical['raw_path'] = canonical.relative_path.map(lambda value: str(CLASS_ROOT/value))
if not canonical.raw_path.map(lambda value: Path(value).is_file()).all(): raise RuntimeError('Raw archive does not match cnn-001 canonical records.')
parts = split_manifest['participants']
for name, selector in {'train': canonical.participant.isin(parts['train']), 'validation': canonical.participant.eq(parts['validation']), 'test': canonical.participant.eq(parts['test'])}.items(): canonical.loc[selector, 'split'] = name
if canonical.split.isna().any(): raise RuntimeError('Participant split reconstruction failed.')
print(canonical.split.value_counts().to_dict())


In [ ]:
# Two-hand, handedness-aware landmark feature: left 63 + right 63 + presence flags + relative wrist xy.
options = mp.tasks.vision.HandLandmarkerOptions(base_options=mp.tasks.BaseOptions(model_asset_path=str(task_path)), running_mode=mp.tasks.vision.RunningMode.IMAGE, num_hands=NUM_HANDS, min_hand_detection_confidence=.5, min_hand_presence_confidence=.5, min_tracking_confidence=.5)
landmarker = mp.tasks.vision.HandLandmarker.create_from_options(options)
def normalized_hand(points):
    array = np.array([[point.x, point.y, point.z] for point in points], dtype=np.float32)
    wrist = array[0].copy(); scale = max(float(np.linalg.norm(array[:, :2] - wrist[:2], axis=1).max()), 1e-6)
    return ((array - wrist) / scale).reshape(-1), wrist, scale
rows = []
try:
    for index, row in enumerate(canonical.itertuples(), 1):
        image = cv2.imread(row.raw_path); status = 'ok'; left = np.zeros(63, dtype=np.float32); right = np.zeros(63, dtype=np.float32); present = np.zeros(2, dtype=np.float32); wrists = {}
        if image is None: status = 'unreadable'
        else:
            rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB); result = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
            if not result.hand_landmarks: status = 'no_hand_detected'
            else:
                assigned = {}
                for hand, handedness in zip(result.hand_landmarks, result.handedness):
                    category = handedness[0].category_name.lower() if handedness else 'unknown'
                    score = float(handedness[0].score) if handedness else 0.0
                    slot = 'left' if category == 'left' else 'right' if category == 'right' else None
                    if slot is not None and (slot not in assigned or score > assigned[slot][0]): assigned[slot] = (score, hand)
                for slot, (_, hand) in assigned.items():
                    features, wrist, scale = normalized_hand(hand); wrists[slot] = (wrist, scale)
                    if slot == 'left': left = features; present[0] = 1
                    else: right = features; present[1] = 1
                if not assigned: status = 'unknown_handedness'
        relative_wrist = np.zeros(2, dtype=np.float32)
        if 'left' in wrists and 'right' in wrists: relative_wrist = (wrists['right'][0][:2] - wrists['left'][0][:2]) / max(wrists['left'][1], wrists['right'][1])
        feature = np.concatenate([left, right, present, relative_wrist])
        record = {'relative_path': row.relative_path, 'label': row.label, 'participant': row.participant, 'split': row.split, 'sha256': row.sha256, 'status': status, 'hands_detected': int(present.sum())}
        record.update({f'f_{i:03d}': float(value) for i, value in enumerate(feature)}); rows.append(record)
        if index % 1000 == 0: print(f'Landmarked {index:,}/{len(canonical):,}', flush=True)
finally:
    landmarker.close()
landmarks = pd.DataFrame(rows); landmarks.to_parquet(OUTPUTS/'metadata'/'landmark_manifest.parquet', index=False)
landmarks.to_csv(OUTPUTS/'metadata'/'landmark_status.csv', columns=['relative_path', 'label', 'participant', 'split', 'sha256', 'status', 'hands_detected'], index=False)
detection_summary = {'canonical_samples': int(len(landmarks)), 'status_counts': {key: int(value) for key, value in landmarks.status.value_counts().items()}, 'split_detection_counts': landmarks.groupby(['split', 'status']).size().unstack(fill_value=0).to_dict()}
(OUTPUTS/'metadata'/'detection_summary.json').write_text(json.dumps(detection_summary, indent=2), encoding='utf-8'); print(json.dumps(detection_summary, indent=2))


In [ ]:
# Tune only C on validation; all sample exclusions are explicit in detection artifacts.
feature_columns = [f'f_{i:03d}' for i in range(130)]
usable = landmarks[landmarks.status == 'ok'].copy()
partitions = {name: usable[usable.split == name].copy() for name in ('train', 'validation', 'test')}
if min(len(frame) for frame in partitions.values()) == 0: raise RuntimeError('A split has no MediaPipe-detected samples.')
label_index = {label: index for index, label in enumerate(CLASSES)}
scores = []; best_model, best_c, best_score = None, None, -1.0
for c in (0.1, 1.0, 10.0):
    candidate = Pipeline([('scaler', StandardScaler()), ('svm', SVC(C=c, kernel='rbf', gamma='scale', cache_size=4000))])
    candidate.fit(partitions['train'][feature_columns], partitions['train'].label.map(label_index)); score = candidate.score(partitions['validation'][feature_columns], partitions['validation'].label.map(label_index)); scores.append({'C': c, 'validation_accuracy': score}); print(scores[-1])
    if score > best_score: best_model, best_c, best_score = candidate, c, score
pd.DataFrame(scores).to_csv(OUTPUTS/'logs'/'validation_search.csv', index=False)
joblib.dump(best_model, OUTPUTS/'models'/'mp_svm_001.joblib')
truth = partitions['test'].label.map(label_index).to_numpy(); predicted = best_model.predict(partitions['test'][feature_columns])
report = classification_report(truth, predicted, labels=range(36), target_names=CLASSES, output_dict=True, zero_division=0); matrix = confusion_matrix(truth, predicted, labels=range(36))
pd.DataFrame(report).T.to_csv(OUTPUTS/'metrics'/'classification_report.csv'); pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(OUTPUTS/'metrics'/'confusion_matrix.csv')
mapping = {label: index for index, label in enumerate(CLASSES)}; o, zero = mapping['O'], mapping['0']
summary = {'experiment_id': EXPERIMENT_ID, 'test_accuracy_detected_subset': float(accuracy_score(truth, predicted)), 'macro_precision': report['macro avg']['precision'], 'macro_recall': report['macro avg']['recall'], 'macro_f1': report['macro avg']['f1-score'], 'O_recall': report['O']['recall'], '0_recall': report['0']['recall'], 'O_to_0': int(matrix[o, zero]), '0_to_O': int(matrix[zero, o]), 'best_validation_accuracy': best_score, 'selected_C': best_c, 'detected_counts': {name: int(len(frame)) for name, frame in partitions.items()}}
(OUTPUTS/'metrics'/'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
plt.figure(figsize=(16,13)); sns.heatmap(matrix, cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.savefig(OUTPUTS/'figures'/'confusion_matrix.png', dpi=180); plt.close()
config = {'experiment_id': EXPERIMENT_ID, 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'dataset_repo': DATASET_REPO, 'dataset_revision': DATASET_REVISION, 'raw_archive': RAW_ARCHIVE, 'raw_archive_sha256': cnn_config['audit_manifest']['raw_archive_sha256'], 'cnn_artifact_repo': CNN_ARTIFACT_REPO, 'cnn_artifact_revision': CNN_ARTIFACT_REVISION, 'mediapipe': {'task_url': MEDIAPIPE_TASK_URL, 'task_sha256': task_sha256, 'num_hands': NUM_HANDS, 'confidence_thresholds': .5, 'feature_schema': 'left 63 + right 63 + two presence flags + relative wrist xy = 130'}, 'split_manifest': split_manifest, 'classifier': {'type': 'StandardScaler + RBF SVC', 'C_candidates': [0.1, 1.0, 10.0], 'selected_C': best_c}, 'classes': CLASSES}
(OUTPUTS/'metadata'/'experiment_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8'); (OUTPUTS/'metadata'/'split_manifest.json').write_text(json.dumps(split_manifest, indent=2), encoding='utf-8'); (OUTPUTS/'metadata'/'deduplication_manifest.csv').write_text((CNN_ROOT/'metadata'/'deduplication_manifest.csv').read_text(), encoding='utf-8')
os.system(f"pip freeze > {OUTPUTS/'metadata'/'environment.txt'}")
print(json.dumps(summary, indent=2))


## Publish

Publish `landmark_manifest.parquet`, status/config to the dataset cache for reuse. Publish model, metrics, notebook and all provenance to `hnam25/asl-hg-mediapipe-svm-baseline`.
